# Stochastic random walk kernels

## Imports

In [1]:
import sys
import os

# Добавляем родительскую папку в пути поиска
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
import numpy as np
import networkx as nx
import scipy.linalg as la
import time

from src import utils
from src import rwk
from src import gvoys
from src import mcrwk

In [3]:
from torch_geometric.datasets import TUDataset
from torch_geometric.utils import to_dense_adj

/Users/marioauditore/Desktop/HSE/Kernels/random_walk_kernels_computations/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset = TUDataset(root='./data', name='MUTAG', use_node_attr=False, use_edge_attr=False)

Processing...
Done!


## Existing Optimizations

Sylvester Equation, Fixed Point and Conjugate Gradient methods from Vishwanathan et al. 2010 are applied only for geometric kernel

#### Unlabeled test

In [3]:
n = 50
G1 = utils.graph_generator(n, kind="er", seed=1)
G2 = utils.graph_generator(n, kind="er", seed=2)

P1 = utils.normalized_adj_matrix(G1)
P2 = utils.normalized_adj_matrix(G2)

v1 = utils.uniform_dist(n)
v2 = utils.uniform_dist(n)
w1 = utils.uniform_dist(n)
w2 = utils.uniform_dist(n)

mu_exp = utils.mu_func_gen("exp", lmbd=0.1)
mu_geom = utils.mu_func_gen("geom", lmbd=0.1)

#exp
k_exp_exact = rwk.random_walk_kernel(P1, P2, v1, v2, w1, w2, mu_exp, kind="exp")
k_exp_trunc = rwk.random_walk_kernel(P1, P2, v1, v2, w1, w2, mu_exp, kind="general", max_iter=20)
print("exp exact :", k_exp_exact)
print("exp trunc :", k_exp_trunc)

#geom
k_geom_exact = rwk.random_walk_kernel(P1, P2, v1, v2, w1, w2, mu_geom, kind="geom")
k_geom_trunc = rwk.random_walk_kernel(P1, P2, v1, v2, w1, w2, mu_geom, kind="general", max_iter=20)

k_geom_sylv = rwk.random_walk_kernel_sylvester(P1, P2, v1, v2, w1, w2, mu_geom)
k_geom_fp = rwk.random_walk_kernel_fixed_point(P1, P2, v1, v2, w1, w2, mu_geom)
k_geom_cg = rwk.random_walk_kernel_cg(P1, P2, v1, v2, w1, w2, mu_geom)

print("geom exact:", k_geom_exact)
print("geom trunc:", k_geom_trunc)
print("sylvester:", k_geom_sylv)
print("fixed point:", k_geom_fp)
print("cg:", k_geom_cg)

exp exact : 0.00044206836723026193
exp trunc : 0.00044206836723026193
geom exact: 0.00044444444444444663
geom trunc: 0.00044444444444444663
sylvester: 0.0004444444444444444
fixed point: 0.00044444444444444447
cg: 0.00044444444444444663


#### Labeled test

In [4]:
# Labeled
n = 50
n_labels = 3
G1 = utils.graph_generator_labeled(n, kind="er", n_labels=n_labels, seed=1)
G2 = utils.graph_generator_labeled(n, kind="er", n_labels=n_labels, seed=2)

P1_labeled = utils.normalized_adj_matrix_labeled(G1)
P2_labeled = utils.normalized_adj_matrix_labeled(G2)

v1 = utils.uniform_dist(n)
v2 = utils.uniform_dist(n)
w1 = utils.uniform_dist(n)
w2 = utils.uniform_dist(n)

mu_exp = utils.mu_func_gen("exp", lmbd=0.1)
mu_geom = utils.mu_func_gen("geom", lmbd=0.1)

k_exp_exact = rwk.random_walk_kernel_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_exp, kind="exp")
k_exp_trunc = rwk.random_walk_kernel_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_exp, kind="general", max_iter=20)
print("exp exact :", k_exp_exact)
print("exp trunc :", k_exp_trunc)

k_geom_exact = rwk.random_walk_kernel_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_geom, kind="geom")
k_geom_trunc = rwk.random_walk_kernel_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_geom, kind="general", max_iter=20)
# k_geom_sylv = random_walk_kernel_sylvester_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_geom)
k_geom_fixed_point = rwk.random_walk_kernel_fixed_point_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_geom)
k_geom_cg = rwk.random_walk_kernel_cg_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_geom)

print("geom exact:", k_geom_exact)
print("geom trunc:", k_geom_trunc)
print("fixed point:", k_geom_fixed_point)
print("cg:", k_geom_cg)

exp exact : 0.0004102582068890604
exp trunc : 0.0004102582068890604
geom exact: 0.0004105078680675973
geom trunc: 0.00041050786806759736
fixed point: 0.00041050786806759844
cg: 0.0004105078680675973


### Graph Voyagers 

from Choromanski et al. 2025. Changes were made to tailor it to make algorithm more correct and robust, tailor it for geometric kernel

In [5]:
# Redefine global variables of gvoys
RANDOM_SEED = 180
np.random.seed(RANDOM_SEED)

SIGMA = 0.1
LAMBDA_COEFF = 0.1
P_HALT = 0.2
NB_RANDOM_WALKS = 1000
BIG_NUMBER = 2000

t_variables = np.random.uniform(size=(2 * BIG_NUMBER, 2 * BIG_NUMBER))
g_variables = np.where(np.random.normal(size=(2 * BIG_NUMBER, 2 * BIG_NUMBER)) > 0.0, 1.0, -1.0,)

# Now the expriment setup
N = 30
lmbd = 0.01
nb_random_walks = NB_RANDOM_WALKS
block_size = NB_RANDOM_WALKS

G1 = utils.graph_generator(N, kind="er", seed=1)
G2 = utils.graph_generator(N, kind="er", seed=2)

P1 = utils.normalized_adj_matrix(G1)
P2 = utils.normalized_adj_matrix(G2)

v1 = utils.uniform_dist(N)
v2 = utils.uniform_dist(N)
w1 = utils.uniform_dist(N)
w2 = utils.uniform_dist(N)

mu_exp = utils.mu_func_gen("exp", lmbd=lmbd)
mu_geom = utils.mu_func_gen("geom", lmbd=lmbd)

t0 = time.perf_counter()
exact_exp = rwk.random_walk_kernel(
    P1, P2, v1, v2, w1, w2,
    mu_func=mu_exp,
    kind="exp",
)
t1 = time.perf_counter()

np.random.seed(RANDOM_SEED)
t2 = time.perf_counter()
approx_exp = gvoys.approximate_graph_kernel_value_with_blocks(
    P1, P2, v1, v2, w1, w2,
    anchor_fraction=1.0,
    kind="exponential",
    lambda_coeff=lmbd,
    p_halt=P_HALT,
    nb_random_walks=nb_random_walks,
    block_size=block_size,
)
t3 = time.perf_counter()

abs_err_exp = abs(exact_exp - approx_exp)
rel_err_exp = abs_err_exp / (abs(exact_exp))

print("exp kernel")
print(f"n={N}, lmbd={lmbd}, n_walks={nb_random_walks}, p_halt={P_HALT}")
print(f"exact  : {exact_exp}  time={t1 - t0}s")
print(f"approx : {approx_exp}  time={t3 - t2}s")
print(f"abs err: {abs_err_exp}")
print(f"rel err: {rel_err_exp}")

t4 = time.perf_counter()
exact_geom = rwk.random_walk_kernel(
    P1, P2, v1, v2, w1, w2,
    mu_func=mu_geom,
    kind="geom",
)
t5 = time.perf_counter()

np.random.seed(RANDOM_SEED)
t6 = time.perf_counter()
approx_geom = gvoys.approximate_graph_kernel_value_with_blocks(
    P1, P2, v1, v2, w1, w2,
    anchor_fraction=1.0,
    kind="geometric",
    lambda_coeff=lmbd,
    p_halt=P_HALT,
    nb_random_walks=nb_random_walks,
    block_size=block_size,
)
t7 = time.perf_counter()

abs_err_geom = abs(exact_geom - approx_geom)
rel_err_geom = abs_err_geom / (abs(exact_geom))

print("geom kernel")
print(f"n={N}, lmbd={lmbd}, n_walks={nb_random_walks}, p_halt={P_HALT}")
print(f"exact  : {exact_geom}  time={t5 - t4}s")
print(f"approx : {approx_geom}  time={t7 - t6}s")
print(f"abs err: {abs_err_geom}")
print(f"rel err: {rel_err_geom}")

exp kernel
n=30, lmbd=0.01, n_walks=1000, p_halt=0.2
exact  : 0.0011222779634268549  time=0.025326874980237335s
approx : 0.0011216116275729298  time=2.629622291016858s
abs err: 6.663358539250441e-07
rel err: 0.0005937351312596391
geom kernel
n=30, lmbd=0.01, n_walks=1000, p_halt=0.2
exact  : 0.0011223344556677878  time=0.013616250013001263s
approx : 0.0011284575014493148  time=0.5670316670148168s
abs err: 6.1230457815270294e-06
rel err: 0.005455633791340589


## Monte Carlo Random Walk Kernel

### Unlabeled

In [6]:
# Использование
rng = mcrwk.TaylorGenerator(np.random.PCG64())

# Сравним с обычным пуассоном
lmbd_val = 0.5
my_samples = rng.exp_taylor(lmbd=lmbd_val, size=1000)
np_samples = rng.poisson(lam=lmbd_val, size=1000)

print(f"Среднее (Taylor): {my_samples.mean():.4f}")
print(f"Среднее (NumPy):  {np_samples.mean():.4f}")


Среднее (Taylor): 0.5110
Среднее (NumPy):  0.5040


In [7]:
N = 50
lmbd = 0.01

G1 = utils.graph_generator(N, kind="er", seed=1)
G2 = utils.graph_generator(N, kind="er", seed=2)

P1 = utils.normalized_adj_matrix(G1)
P2 = utils.normalized_adj_matrix(G2)

v1 = utils.uniform_dist(N)
v2 = utils.uniform_dist(N)
w1 = utils.uniform_dist(N)
w2 = utils.uniform_dist(N)

#exp test
mu_exp = utils.mu_func_gen("exp", lmbd=lmbd)
exact_exp = rwk.random_walk_kernel(P1, P2, v1, v2, w1, w2, mu_func=mu_exp, kind="exp")
approx_exp = mcrwk.random_walk_kernel_mc(P1, P2, v1, v2, w1, w2, mu_func=mu_exp, kind="exp", n_samples=1000, seed=42)
print("exact exp :", exact_exp)
print("mc exp:", approx_exp)
print("abs err:", abs(exact_exp - approx_exp))
print(f"rel err: {abs(exact_exp - approx_exp)/(abs(exact_exp))}")

#geom test
mu_geom = utils.mu_func_gen("geom", lmbd=lmbd)
exact_geom = rwk.random_walk_kernel(P1, P2, v1, v2, w1, w2, mu_func=mu_geom, kind="geom")
approx_geom = mcrwk.random_walk_kernel_mc(P1, P2, v1, v2, w1, w2, mu_func=mu_geom, kind="geom", n_samples=1000, seed=42)
print("exact geom :", exact_geom)
print("mc geom:", approx_geom)
print("abs err:", abs(exact_geom - approx_geom))
print(f"rel err: {abs(exact_geom - approx_geom)/(abs(exact_geom))}")

exact exp : 0.0004040200668336694
mc exp: 0.0004040200668336673
abs err: 2.1141942363467336e-18
rel err: 5.232894130521304e-15
exact geom : 0.0004040404040404051
mc geom: 0.0004040404040404042
abs err: 9.215718466126788e-19
rel err: 2.2808903203663736e-15


"linear" time with respect to dataset size& we precompute samples for each graph and then we can compare each pair almost with constant time

### Labeled

In [8]:
# def sample_label_seq(common_labels, q, K, n_label_samples_per_length, rng):
#     label_seqs = []
#     q_prods = np.ones(n_label_samples_per_length, dtype=float)
    
#     for i in range(n_label_samples_per_length):
#         ids = rng.choice(len(common_labels), size=K, p=q)
#         seq = [common_labels[j] for j in ids]
#         label_seqs.append(seq)
#         q_prods[i] = float(np.prod(q[ids]))
    
#     return label_seqs, q_prods

# def prepare_P(P):
#     P_sampling = {}
    
#     for label, P_label in P.items():
#         dist_for_nodes = []
#         for i in range(P_label.shape[0]):
#             row = P_label[i]
#             row_sum = row.sum()
#             if row_sum > 0:
#                 neigh = np.where(row > 0)[0]
#                 probs = row[neigh] / row_sum
#             else:
#                 neigh = np.array([], dtype=int)
#                 probs = np.array([], dtype=float)
#             dist_for_nodes.append(((neigh, probs, row_sum)))
#         P_sampling[label] = dist_for_nodes

#     return P_sampling

# def process_sequence_multi(P_sampling, v, w, label_seq, n_reps, rng):
#     total = 0.0
#     for _ in range(n_reps):
#         x = rng.choice(len(v), p=v)
#         weight = 1.0
#         for label in label_seq:
#             neigh, probs, row_sum = P_sampling[label][x]
#             if row_sum == 0.0:
#                 weight = 0.0
#                 break
#             weight *= row_sum
#             x = rng.choice(neigh, p=probs)
#         total += weight * w[x] if weight != 0.0 else 0.0
#     return total / n_reps


# def build_features_labeled(P, v, w, shared_lengths, shared_label_seqs, shared_q_prods, n_length_samples, n_label_samples_per_length, n_walk_reps, rng):
#     P_sampling = prepare_P(P)
#     features = np.zeros((n_length_samples, n_label_samples_per_length), dtype=float)
#     for i in range(n_length_samples):
#         for j in range(n_label_samples_per_length):
#             curr_seq = shared_label_seqs[i][j]
#             curr_seq_prob = shared_q_prods[i][j]
#             s = process_sequence_multi(P_sampling, v, w, curr_seq, n_walk_reps, rng)
#             features[i, j] = s / math.sqrt(curr_seq_prob)

#     return features


# def q_sampling(P1, P2, d, common_labels, q_sampling_kind="uniform"):
#     if q_sampling_kind == "uniform":
#         return np.ones(d, dtype=float) / d
#     elif q_sampling_kind == "random":
#         x = np.random.random(d)
#         return x / x.sum()
    
#     elif q_sampling_kind == "norm_fro":
#         scores = np.zeros(d, dtype=float)
#         for i, lab in enumerate(common_labels):
#             scores[i] = np.linalg.norm(P1[lab], ord="fro") * np.linalg.norm(P2[lab], ord="fro")
#         if np.all(scores == 0):
#             return np.ones(d, dtype=float) / d
#         return scores / scores.sum()

#     elif q_sampling_kind == "norm_l1":
#         scores = np.zeros(d, dtype=float)
#         for i, lab in enumerate(common_labels):
#             scores[i] = np.sum(np.abs(P1[lab])) * np.sum(np.abs(P2[lab]))
#         if np.all(scores == 0):
#             return np.ones(d, dtype=float) / d
#         return scores / scores.sum()

#     raise ValueError("unknown kind")


# def random_walk_kernel_mc_labeled(P1, P2, v1, v2, w1, w2, mu_func, kind, n_length_samples=200, n_label_samples_per_length=50, n_walk_reps=10, q_sampling_kind="norm_fro", seed=42):
#     rng = np.random.default_rng(seed)
#     C = kernel_normalizer(kind, mu_func)
#     common_labels = sorted(set(P1.keys()) & set(P2.keys()))
#     d = len(common_labels)
#     if d == 0:
#         return 0.0
#     q = q_sampling(P1, P2, d, common_labels, q_sampling_kind=q_sampling_kind)

#     shared_lengths = np.zeros(n_length_samples, dtype=int)
#     shared_label_seqs = []
#     shared_q_prods = []
    
#     for i in range(n_length_samples):
#         K = sample_length(kind, mu_func, rng)
#         shared_lengths[i] = K
#         label_seqs, q_prods = sample_label_seq(common_labels, q, K, n_label_samples_per_length, rng)
#         shared_label_seqs.append(label_seqs)
#         shared_q_prods.append(q_prods)

#     g1 = build_features_labeled(P1, v1, w1, shared_lengths, shared_label_seqs, shared_q_prods, n_length_samples, n_label_samples_per_length, n_walk_reps, rng)
#     g2 = build_features_labeled(P2, v2, w2, shared_lengths, shared_label_seqs, shared_q_prods, n_length_samples, n_label_samples_per_length, n_walk_reps, rng)
#     return C * (g1 * g2).mean(axis=1).mean()

In [9]:
N = 50
n_labels = 3
lmbd = 0.1

n_length_samples = 50 * N
n_label_samples_per_length = 100

G1 = utils.graph_generator_labeled(N, kind="ba", n_labels=n_labels, seed=1)
G2 = utils.graph_generator_labeled(N, kind="ba", n_labels=n_labels, seed=2)

P1 = utils.normalized_adj_matrix_labeled(G1)
P2 = utils.normalized_adj_matrix_labeled(G2)

v1 = utils.uniform_dist(N)
v2 = utils.uniform_dist(N)
w1 = utils.uniform_dist(N)
w2 = utils.uniform_dist(N)

mu_geom = utils.mu_func_gen("geom", lmbd=lmbd)

k_direct_geom = rwk.random_walk_kernel_labeled(P1, P2, v1, v2, w1, w2, mu_func=mu_geom, kind="geom")
k_fp_geom = rwk.random_walk_kernel_fixed_point_labeled(P1, P2, v1, v2, w1, w2, mu_func=mu_geom)
k_cg_geom = rwk.random_walk_kernel_cg_labeled(P1, P2, v1, v2, w1, w2, mu_func=mu_geom)
k_mc_geom = mcrwk.random_walk_kernel_mc_labeled(P1, P2, v1, v2, w1, w2, mu_func=mu_geom, kind="geom", 
            n_length_samples=n_length_samples, n_label_samples_per_length=n_label_samples_per_length, n_walk_reps=1, q_sampling_kind="norm_fro", seed=42)
k_mc_geom_reps = mcrwk.random_walk_kernel_mc_labeled(P1, P2, v1, v2, w1, w2, mu_func=mu_geom, kind="geom", 
            n_length_samples=n_length_samples, n_label_samples_per_length=n_label_samples_per_length, n_walk_reps=20, q_sampling_kind="norm_fro", seed=42)

print("labeled geom")
print(f"n={N}, n_labels={n_labels}, lambda={lmbd}")
print(f"direct: {k_direct_geom}")
print(f"fixed_point: {k_fp_geom}")
print(f"cg: {k_cg_geom}")
print(f"our mc: {k_mc_geom}")
print(f"our mc with reps: {k_mc_geom_reps}")
print(f"rel err fp: {abs(k_direct_geom - k_fp_geom)/(abs(k_direct_geom))}")
print(f"rel err cg: {abs(k_direct_geom - k_cg_geom)/(abs(k_direct_geom))}")
print(f"rel err mc: {abs(k_direct_geom - k_mc_geom)/(abs(k_direct_geom))}")
print(f"rel err mc with reps: {abs(k_direct_geom - k_mc_geom_reps)/(abs(k_direct_geom))}")

labeled geom
n=50, n_labels=3, lambda=0.1
direct: 0.00041406517812670406
fixed_point: 0.0004140651781267047
cg: 0.00041406517812670444
our mc: 0.0004134155131797898
our mc with reps: 0.00041340456800612204
rel err fp: 1.5710601563608011e-15
rel err cg: 9.16451757877134e-16
rel err mc: 0.0015689919878155937
rel err mc with reps: 0.0015954254438171494


In [10]:
def random_walk_kernel_mc_dataset_labeled():
    ...

## Benchmarks

### Unlabeled

In [11]:
def bench(dataset, name, mu_func, n_graphs, n_samples_mc, n_samples_gvoys, seed=42):
    ...

### Labeled